# Colab (T4): el repo en una GPU prestada
Lo mismo que los notebooks locales, en la T4 de Colab: clona el repo, baja lo que falta, mide la inferencia con la CLI y copia los JSON a Drive para no perderlos
al cerrar la sesión. Elegí *Entorno de ejecución → Cambiar tipo → T4 GPU*.

## 1. Preparación
Clona el repo si no está (en local no hace nada) e instala lo que falta.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/bpeco/ml2-gen-world-model.git"   # el repo del equipo (en Colab; en local no se usa)
if not any((d / "src" / "common.py").exists() for d in [Path.cwd(), *Path.cwd().parents]):
  if not REPO:
    raise SystemExit("completá REPO con la URL del repo, o abrí el notebook dentro de una copia del repo")
  if not Path("ml2-gen-world-model").exists():
    subprocess.run(["git", "clone", REPO], check=True)
  os.chdir("ml2-gen-world-model")
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
%matplotlib inline
import os, sys
from pathlib import Path

# la raíz del repo: sube desde donde esté abierto el notebook hasta encontrar src/common.py
RAIZ = next((d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "src" / "common.py").exists()), None)
if RAIZ is None:
  raise SystemExit("abrí el notebook dentro del repo (en Colab, corré antes la primera celda de notebooks/colab.ipynb)")
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

from src import nbui
from src.toolkit import Toolkit

# scripts/run_notebooks.py pone ML2_NB_LIMITE para correr todo chico; a mano queda en None (sin recorte)
LIMITE = int(os.environ.get("ML2_NB_LIMITE", "0")) or None

## 2. Dispositivo y datos
Los checkpoints no se distribuyen: cada uno entrena lo que necesita (`python -m src.cli <modelo> entrenar`) o los trae del almacén compartido (`python -m src.store pull <corrida> --dir RUTA`).

In [ ]:
DISP = nbui.backends()
DEVICE = nbui.device_inicial(DISP)
DEV = nbui.device_kw(DEVICE)
nbui.catalogo_corridas();

## 3. Latencia de punta a punta en la T4
Una cadena por modelo, con la CLI (`bench --cadena`); sin checkpoint mide la arquitectura sin entrenar.

In [ ]:
DRY_RUN = bool(os.environ.get("ML2_NB_LIMITE"))
FRAMES = LIMITE or 200
for modelo, run, extra in (("genie", "genie_s", dict(pasos="4")), ("convlstm_px", "convlstm_px", {}), ("lewm", "lewm", {})):
  nbui.lanzar(modelo, "bench", dry_run=DRY_RUN, segundo_plano=False, cadena=True, ritmo=50, limite=FRAMES, run=run, aleatorio=True, **extra, **DEV)
nbui.latencia_etapas();

## 4. Calidad en línea y con la cadena
`evaluar --cadena` mide la calidad con el codec real en el medio (necesita los checkpoints).

In [ ]:
for modelo, run in (("genie", "genie_s"), ("convlstm_px", "convlstm_px")):
  nbui.lanzar(modelo, "evaluar", dry_run=True, cadena=True, run=run, limite=LIMITE, **DEV)
nbui.tabla_video();

## 5. Copiar los JSON a Drive
Todo `results/` (sin los cachés pesados) a tu Drive, para llevarlo al repo local. En local no hace nada.

In [ ]:
try:
  from google.colab import drive
  drive.mount("/content/drive")
  import shutil
  destino = Path("/content/drive/MyDrive/ml2-gen-world-model/results")
  shutil.copytree("results", destino, dirs_exist_ok=True, ignore=shutil.ignore_patterns("cache", "rollouts", "*.npy"))
  print("copiado a", destino)
except ImportError:
  print("no es Colab: results/ ya está en el repo local")